In [2]:
import kagglehub

spaceship_titanic_path = kagglehub.competition_download('spaceship-titanic')

print('Data source import complete.')


100%|██████████| 299k/299k [00:00<00:00, 65.0MB/s]

Extracting files...
Data source import complete.


In [3]:
print(spaceship_titanic_path)

/root/.cache/kagglehub/competitions/spaceship-titanic


In [4]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/content/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/content/.config/gce
/content/.config/.last_opt_in_prompt.yaml
/content/.config/.last_survey_prompt.yaml
/content/.config/active_config
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/default_configs.db
/content/.config/.last_update_check.json
/content/.config/config_sentinel
/content/.config/logs/2026.06.04/13.32.02.654775.log
/content/.config/logs/2026.06.04/13.31.42.499627.log
/content/.config/logs/2026.06.04/13.32.39.344962.log
/content/.config/logs/2026.06.04/13.32.18.735754.log
/content/.config/logs/2026.06.04/13.32.38.346437.log
/content/.config/logs/2026.06.04/13.32.21.210570.log
/content/.config/configurations/config_default
/content/sample_data/anscombe.json
/content/sample_data/README.md
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_test.csv


Viewing the dataset, to look at the kinds of values in the set

In [5]:
train_df = pd.read_csv(f"{spaceship_titanic_path}/train.csv")
test_df = pd.read_csv(f"{spaceship_titanic_path}/test.csv")


train_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [6]:
train_df.shape

(8693, 14)

Checking for null values

In [7]:
(train_df.isnull().sum() / len(train_df)) * 100

,0
PassengerId,0.000000
HomePlanet,2.312205
CryoSleep,2.496261
Cabin,2.289198
Destination,2.093639
Age,2.059128
VIP,2.335212
RoomService,2.082135
FoodCourt,2.105142
ShoppingMall,2.392730


Checking if any of the categorical columns have spelling errors or non uniform spellings (Earth and earth for instance)

In [12]:
categorical = train_df.select_dtypes(include="object").columns

for col in categorical:
  print(col, train_df[col].unique(), "\n")

PassengerId ['0001_01' '0002_01' '0003_01' ... '9279_01' '9280_01' '9280_02'] 

HomePlanet ['Europa' 'Earth' 'Mars' nan] 

CryoSleep [False True nan] 

Cabin ['B/0/P' 'F/0/S' 'A/0/S' ... 'G/1499/S' 'G/1500/S' 'E/608/S'] 

Destination ['TRAPPIST-1e' 'PSO J318.5-22' '55 Cancri e' nan] 

VIP [False True nan] 

Name ['Maham Ofracculy' 'Juanna Vines' 'Altark Susent' ... 'Fayey Connon'
 'Celeon Hontichre' 'Propsh Hontichre'] 



Although a lot of the spending columns have null values, they might be related to people who are in cryosleep. If someone is in cryosleep and the spending column is null, that is probably fine and can be replaced by 0's

In [8]:
spending_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

In [9]:
train_df.groupby('CryoSleep')[spending_cols].apply(lambda x: x.isnull().sum())

,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
CryoSleep,,,,,
False,111,110,104,113,119
True,68,70,96,65,62


In [10]:
train_df[train_df['CryoSleep'] == True][spending_cols].describe() ##Checking if my assumption of spending 0 is true

,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,2969.0,2967.0,2941.0,2972.0,2975.0
mean,0.0,0.0,0.0,0.0,0.0
std,0.0,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0,0.0
75%,0.0,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0,0.0


I must first deal with all the rows where cryosleep is null, before I can apply the spending_cols to 0 rule

In [18]:
train_df["CryoSleep"].value_counts()

,count
CryoSleep,
False,5439
True,3037


In [21]:
known = train_df[train_df['CryoSleep'].notnull()].copy()
known['total_spend'] = known[spending_cols].sum(axis=1)
pd.crosstab(known['CryoSleep'], known['total_spend'] == 0)

total_spend,False,True
CryoSleep,,
False,4921,518
True,0,3037


From the above, we can infer that if cryosleep is true, then spending will always be 0

If cryosleep is false spending can be either true or false

using this I can fill in the cryosleep null values to be True wherever the total_spend is > 0

In [25]:
spend_sum = train_df[spending_cols].sum(axis=1)
cryo_nulls = train_df["CryoSleep"].isnull()

train_df.loc[cryo_nulls & (spend_sum > 0), "CryoSleep"] = False

In [26]:
(train_df.isnull().sum() / len(train_df)) * 100

,0
PassengerId,0.000000
HomePlanet,2.312205
CryoSleep,1.127344
Cabin,2.289198
Destination,2.093639
Age,2.059128
VIP,2.335212
RoomService,2.082135
FoodCourt,2.105142
ShoppingMall,2.392730


The cryosleep columns with a missing vaue went down from 2.49% to 1.12%

In [27]:
train_df["CryoSleep"] = train_df["CryoSleep"].fillna("unkown")

Now I will change all the spending column NaN values to 0's wherever CryoSleep is true

In [29]:
train_df.loc[train_df["CryoSleep"] == True, spending_cols] = train_df.loc[train_df["CryoSleep"] == True, spending_cols].fillna(0)

In [36]:
train_df[spending_cols].describe()

,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8580.000000,8580.000000,8581.000000,8575.000000,8567.000000
mean,222.906876,454.339977,171.785573,308.780292,302.648535
std,664.368930,1605.430304,601.581625,1132.710170,1141.855811
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,44.000000,70.000000,24.000000,58.000000,44.000000
max,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [37]:
train_df[spending_cols].median()

,0
RoomService,0.0
FoodCourt,0.0
ShoppingMall,0.0
Spa,0.0
VRDeck,0.0


Turns out the median value of spend is 0, I could've just done fillna(0) directly

In [38]:
train_df[spending_cols] = train_df[spending_cols].fillna(0)

In [39]:
train_df.isnull().sum()

,0
PassengerId,0
HomePlanet,201
CryoSleep,0
Cabin,199
Destination,182
Age,179
VIP,203
RoomService,0
FoodCourt,0
ShoppingMall,0


The data card says that the passengerId is split into group_id and passenger_id, I can split this into two new columns and drop the passengerId column

In [42]:
train_df[["group_id", "passenger_id"]] = train_df["PassengerId"].astype(str).str.split("_", expand=True)
train_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,group_id,passenger_id
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,0001,01
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,0002,01
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,0003,01
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,0003,02
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,0004,01


I can safely drop the passengerId column

In [ ]:
train_df = train_df.drop(columns=["PassengerId"])

Similarly I can split the Cabin into deck, number and side

In [49]:
train_df[["deck", "number", "side"]] = train_df["Cabin"].astype(str).str.split("/", expand=True)

In [50]:
train_df.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,group_id,passenger_id,deck,number,side
0,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,0001,01,B,0,P
1,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,0002,01,F,0,S
2,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,0003,01,A,0,S
3,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,0003,02,A,0,S
4,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,0004,01,F,1,S


I can now safely drop the Cabin column

In [51]:
train_df = train_df.drop(columns=["Cabin"])

In [52]:
train_df.isnull().sum()

,0
HomePlanet,201
CryoSleep,0
Destination,182
Age,179
VIP,203
RoomService,0
FoodCourt,0
ShoppingMall,0
Spa,0
VRDeck,0


In [57]:
train_df[['deck','number','side']].isnull().sum()

,0
deck,0
number,199
side,199


In [58]:
train_df[train_df['number'].isnull()][['deck','number','side']].head(10)

,deck,number,side
15,nan,None,None
93,nan,None,None
103,nan,None,None
222,nan,None,None
227,nan,None,None
251,nan,None,None
260,nan,None,None
272,nan,None,None
280,nan,None,None
295,nan,None,None


In [59]:
train_df['deck'] = train_df['deck'].replace('nan', np.nan)

In [61]:
train_df.isnull().sum()

,0
HomePlanet,201
CryoSleep,0
Destination,182
Age,179
VIP,203
RoomService,0
FoodCourt,0
ShoppingMall,0
Spa,0
VRDeck,0


In [62]:
group_deck = train_df.dropna(subset=['deck']).groupby('group_id')['deck'].nunique()
group_side = train_df.dropna(subset=['side']).groupby('group_id')['side'].nunique()

print("Fraction of groups with 1 unique deck:", (group_deck == 1).mean())
print("Fraction of groups with 1 unique side:", (group_side == 1).mean())

Fraction of groups with 1 unique deck: 0.9311866623079438
Fraction of groups with 1 unique side: 1.0


In [65]:
group_sizes = train_df.groupby('group_id')['passenger_id'].count()
multi_groups = group_sizes[group_sizes > 1].index

multi_deck = train_df[train_df['group_id'].isin(multi_groups)].dropna(subset=['deck']).groupby('group_id')['deck'].nunique()
multi_side = train_df[train_df["group_id"].isin(multi_groups)].dropna(subset=["side"]).groupby("group_id")["side"].nunique()
print("Fraction of MULTI-passenger groups sharing 1 deck:", (multi_deck == 1).mean())
print("Fraction of MULTI-passenger groups sharing 1 side:", (multi_side == 1).mean())

Fraction of MULTI-passenger groups sharing 1 deck: 0.7018413597733711
Fraction of MULTI-passenger groups sharing 1 side: 1.0


I can impute that all the passengers in a group will be in the same side, although the passengers in a group are only on the same deck 70% of the time, this is good enough for imputing the values, since only about 2% of the data is NaN

In [66]:
group_side_map = train_df.dropna(subset=['side']).groupby('group_id')['side'].agg(lambda x: x.mode()[0])

mask = train_df['side'].isnull()
train_df.loc[mask, 'side'] = train_df.loc[mask, 'group_id'].map(group_side_map)

train_df['side'].isnull().sum()

np.int64(99)

In [67]:
group_deck_map = train_df.dropna(subset=['deck']).groupby('group_id')['deck'].agg(lambda x: x.mode()[0])

mask_group = train_df['deck'].isnull()
train_df.loc[mask_group, 'deck'] = train_df.loc[mask_group, 'group_id'].map(group_deck_map)

train_df['deck'].isnull().sum()

np.int64(99)

All of the remaining passengers are travelling alone, there's nothing I can impute, so I'll fill in their null values using unknown

In [69]:
train_df["deck"] = train_df["deck"].fillna("unknown")
train_df["side"] = train_df["side"].fillna("unknown")
train_df["number"] = train_df["number"].fillna(-1)

In [70]:
train_df.isnull().sum()

,0
HomePlanet,201
CryoSleep,0
Destination,182
Age,179
VIP,203
RoomService,0
FoodCourt,0
ShoppingMall,0
Spa,0
VRDeck,0


In [71]:
train_df.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,group_id,passenger_id,deck,number,side
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,0001,01,B,0,P
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,0002,01,F,0,S
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,0003,01,A,0,S
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,0003,02,A,0,S
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,0004,01,F,1,S


In [72]:
train_df['Surname'] = train_df['Name'].str.split(' ').str[-1]

In [76]:
placeholder = pd.Series('Unknown_' + train_df.index.astype(str), index=train_df.index)
train_df['Surname'] = train_df['Surname'].fillna(placeholder)

In [77]:
train_df.isnull().sum()

,0
HomePlanet,201
CryoSleep,0
Destination,182
Age,179
VIP,203
RoomService,0
FoodCourt,0
ShoppingMall,0
Spa,0
VRDeck,0


In [78]:
train_df = train_df.drop(columns=["Name"])

I can most likely guess that all the passengers in one group started at the same source and might end up going to the same destination

In [80]:
multi_groups = train_df.groupby('group_id')['passenger_id'].transform('count') > 1
multi_hp = train_df[multi_groups].dropna(subset=['HomePlanet']).groupby('group_id')['HomePlanet'].nunique()
print("Fraction of multi-passenger groups sharing 1 HomePlanet:", (multi_hp == 1).mean())

Fraction of multi-passenger groups sharing 1 HomePlanet: 1.0


In [81]:
multi_dest = train_df[multi_groups].dropna(subset=['Destination']).groupby('group_id')['Destination'].nunique()
print("Fraction of multi-passenger groups sharing 1 Destination:", (multi_dest == 1).mean())

Fraction of multi-passenger groups sharing 1 Destination: 0.4922096317280453


Clearly, the fraction of passengers sharing the same destination is not reliable enough to impute the value, I will impute the value for source and fill in unknown for destination

In [82]:
group_hp_map = train_df.dropna(subset=['HomePlanet']).groupby('group_id')['HomePlanet'].agg(lambda x: x.mode()[0])
mask = train_df['HomePlanet'].isnull()
train_df.loc[mask, 'HomePlanet'] = train_df.loc[mask, 'group_id'].map(group_hp_map)

In [83]:
train_df["Destination"] = train_df["Destination"].fillna("unknown")

In [85]:
train_df["HomePlanet"] = train_df["HomePlanet"].fillna("unknown")

In [86]:
train_df.isnull().sum()

,0
HomePlanet,0
CryoSleep,0
Destination,0
Age,179
VIP,203
RoomService,0
FoodCourt,0
ShoppingMall,0
Spa,0
VRDeck,0


In [88]:
train_df["VIP"].value_counts()

,count
VIP,
False,8291
True,199


In [91]:
train_df["VIP"] = train_df["VIP"].fillna("unknown")
train_df["Age"] = train_df["Age"].fillna(train_df["Age"].median())

In [92]:
train_df.duplicated().sum()

np.int64(0)

In [93]:
train_df[['Age'] + spending_cols].describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8693.000000,8693.000000,8693.000000,8693.000000,8693.000000,8693.000000
mean,28.790291,220.009318,448.434027,169.572300,304.588865,298.261820
std,14.341404,660.519050,1595.790627,598.007164,1125.562559,1134.126417
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,20.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,37.000000,41.000000,61.000000,22.000000,53.000000,40.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [94]:
train_df[["Age"] + spending_cols].median()

,0
Age,27.0
RoomService,0.0
FoodCourt,0.0
ShoppingMall,0.0
Spa,0.0
VRDeck,0.0


In [95]:
train_df.columns

Index(['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService',
       'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Transported', 'group_id',
       'passenger_id', 'deck', 'number', 'side', 'Surname'],
      dtype='object')

In [97]:
train_df['CryoSleep'] = train_df['CryoSleep'].replace('unkown', 'unknown')

In [99]:
train_df['CryoSleep'] = train_df['CryoSleep'].astype(str)
train_df['CryoSleep'].unique()

array(['False', 'True', 'unknown'], dtype=object)

In [102]:
cols_to_fix = ["CryoSleep", "VIP"] # Ensure these are your exact columns

# 1. Create a dictionary that catches strings, actual booleans, and your "unknown" text
bool_mapping = {
    'True': True,
    'False': False,
    True: True,
    False: False,
    'unknown': pd.NA  # pd.NA is pandas' native missing value for the boolean type
}

# 2. Apply the map and cast to boolean
for col in cols_to_fix:
    train_df[col] = train_df[col].map(bool_mapping).astype("boolean")

train_df[cols_to_fix].head()

,CryoSleep,VIP
0,False,False
1,False,False
2,False,True
3,False,False
4,False,False


In [105]:
train_df["number"] = train_df["number"].astype(np.float64)

In [106]:
train_df.to_parquet('train_cleaned.parquet')